# 08 · Circulación de gran escala

¿El viento en altura, la convección amazónica y la MJO añaden algo a lo que ya saben la lluvia local, el suelo, el caudal y el mar?

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Es un análisis de investigación con un catálogo de fechas incompleto. Ante una emergencia llama al 115 (INDECI).

Las hipótesis, las cajas, los rezagos y el criterio de éxito se escribieron y subieron **antes** de descargar estas variables (`docs/08-circulacion.md`, sección 1):

| | Hipótesis | Variable | Rezagos |
|---|---|---|---|
| T16 | Viento del este en 200 hPa sobre el Altiplano distingue los días con huaico | `u200_altiplano` (20S-12S, 75W-65W) | 1 a 5 d |
| T17 | OLR bajo en la Amazonía y la fase de la MJO modulan | `olr_amazonia` (15S-5S, 70W-55W), RMM, OMI | 5 a 15 d |
| T18 | El SOI añade algo más allá del mar local | SOI del mes anterior | mes anterior |
| T19 | El flujo de humedad del este sobre la selva central precede | `u850/v850/q850_selva` (14S-8S, 72W-66W) | 1 a 5 d |
| T20 | Una vaguada o baja segregada en 500 hPa frente a Chile-Perú precede | `z500_chile_peru` (30S-15S, 85W-72W) y `z500_vaguada` | 0 a 4 d (≥ 1 en el modelo) |

Éxito: el AP fuera de muestra de `lluvia+antecedentes` sube más que el percentil 95 de lo que ganan las mismas columnas con las fechas desplazadas dentro de cada temporada, y se mantiene dejando episodios Niño fuera.

In [ ]:
from pathlib import Path

import pandas as pd

from illapa_research import circulacion as circ
from illapa_research import modelo_diario as md
from illapa_research import senales as S

N_PERM = 1000  # permutaciones de la nula por (variable, rezago)
RAIZ = Path("..") if Path("../datos").exists() else Path(".")
D = md.cargar_datos(RAIZ)
Cir = circ.construir(RAIZ / "datos", D.F.index)
print(D.inventario["ultimo_dia"], "| columnas de circulación:", list(Cir.columns))
Cir.describe().T[["count", "mean", "std"]].round(2)

## 1. Rezagos 1 a 15 días: ¿qué distingue los días con huaico?

AUC de cada variable en cada rezago contra una nula que desplaza las etiquetas dentro de la temporada, con Benjamini-Hochberg (q = 0,10). `centrar=True` quita la media de cada temporada (queda solo el momento dentro de la temporada).

In [ ]:
F = pd.concat([D.F, Cir], axis=1)
mask = D.inventario["mask"]
cols = list(Cir.columns)
res = {}
for centrar in (False, True):
    r = S.rezagos_nula(F, D.y, cols, D.temporada, mask, lags=range(1, 16), centrar=centrar, n_null=N_PERM)
    res[centrar] = r
    print("centrado por temporada" if centrar else "sin centrar", "| pruebas:", r.attrs["n_pruebas"],
          "| pasan BH:", r.attrs["observados"], "| p<0,05:", r.attrs["observados_p05"],
          "| esperadas por azar:", round(r.attrs["esperados_p05_por_azar"], 1))
r = res[True]
r[r.senal].sort_values("p").head(25).round(3)

In [ ]:
(res[True].pivot(index="variable", columns="lag", values="auc") - 0.5).round(2)

## 2. Robustez de lo que pasa el filtro

In [ ]:
masc = S.mascaras_robustez(F, D.temporada, mask, D.temporadas)
rob = S.robustez(F, D.y, cols, D.temporada, masc, lags=range(1, 16), centrar=True, n_null=N_PERM)
print("pruebas:", rob.attrs["n_pruebas"], "| robustas:", int(rob["robusta"].sum()))
rob[rob["robusta"]].round(3)

## 3. ¿Mejoran el modelo diario? (el criterio de éxito)

Se agrega cada bloque, desplazado un día, a `lluvia+antecedentes` y se mide la ganancia de AP fuera de muestra (regresión logística con L2 fija, C = 0,03, en la GPU por lotes). **Nula:** las filas del bloque se mueven cíclicamente dentro de cada temporada y se vuelve a ajustar cada pliegue, 2 000 veces. Pliegues: una temporada fuera y un episodio Niño fuera.

Esta corrida es pesada: se hizo en la PC remota con GPU y el resultado está en `docs/08-resultados/gpu_resultados.csv`. Con `RECALCULAR = True` se rehace aquí (en CPU tarda horas con 2 000 réplicas; para probar, baja `N_NULAS`).

In [ ]:
RECALCULAR = False
N_NULAS = 200
CSV = RAIZ / "docs" / "08-resultados" / "gpu_resultados.csv"
if RECALCULAR or not CSV.exists():
    G = circ.grupos(cols)
    res_gpu = circ.correr_gpu(RAIZ, [{"prueba": k, "columnas": v} for k, v in G.items()], n_nulas=N_NULAS)
    res_gpu["escenario"] = "principal (recalculado)"
else:
    res_gpu = pd.read_csv(CSV)
res_gpu["supera p95"] = res_gpu["ganancia"] > res_gpu["nula_p95"]
vista = ["escenario", "prueba", "pliegues", "ap_base", "ap_con", "ganancia", "nula_p95", "nula_p99", "p", "n_nulas", "supera p95"]
res_gpu.query("escenario.str.startswith('principal') and ~prueba.str.startswith('T19:')")[vista].round(4)

### Ablación de T19 (una variable a la vez)

In [ ]:
res_gpu[res_gpu.prueba.str.startswith("T19:")].query("escenario.str.startswith('principal')")[vista].round(4)

### Robustez: etiqueta del día anterior, sin 1983/1998/2017, desde 2009 y hasta 2022 (sin imputar el OLR)

In [ ]:
res_gpu[~res_gpu.escenario.str.startswith("principal")][vista].round(4)

## Lectura

Ver `docs/08-circulacion.md` para la interpretación con las cifras de la corrida, los datos usados y los límites. Las variables de gran escala vienen de reanálisis de malla gruesa (NCEP/NCAR R1, 2,5°): describen el patrón regional, no lo que pasa en una quebrada.